In [17]:
path = "/itet-stor/bszarvas/net_scratch/cache"
data_path = "gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr"

from dinosaur import coordinate_systems, filtering, primitive_equations, spherical_harmonic, sigma_coordinates, scales
import numpy as np
import os
import xarray as xr



def attach_data_array_units(array):
    attrs = dict(array.attrs)
    units = attrs.pop('units', None)
    if units in {'(0-1)', '%', '~'}:
        units = None
    if units is not None:
        data = scales.units.parse_expression(units) * array.data
    else:
        data = scales.units.dimensionless * array.data
    return xr.DataArray(data, array.coords, array.dims, attrs=attrs)


def xarray_nondimensionalize(ds):
    return xr.apply_ufunc(scales.DEFAULT_SCALE.nondimensionalize, ds)

ds = xr.open_zarr(data_path, chunks=None, storage_options=dict(token='anon'))

geopotential_at_surface = ds.geopotential_at_surface

target_coords = spherical_harmonic.Grid.TL127() ###SETNGRID SIZE HERE

interpolated_geopot = ds.geopotential_at_surface.interp(latitude=target_coords.latitudes, longitude=target_coords.longitudes)

geopot_with_units = attach_data_array_units(interpolated_geopot)

orography = geopot_with_units / scales.GRAVITY_ACCELERATION

orography_nondim = xarray_nondimensionalize(orography)

print(orography_nondim.values.flat[:5])


orography_final = target_coords.to_modal(orography_nondim)

filter_fn = filtering.exponential_filter(target_coords, order=2)

filtered_orography = filter_fn(orography_final)

print(filtered_orography.shape)

os.makedirs(path, exist_ok=True)


np.savez_compressed(os.path.join(path, f"era5_model_{target_coords.modal_shape[0]}_13.npz"), filtered_orography=filtered_orography)

[-4.32384637e-09 -4.36906603e-09 -4.41455274e-09 -4.37210919e-09
 -4.31514840e-09]
(255, 129)


Check GPU

In [11]:
import xarray as xr

path = '/itet-stor/bszarvas/net_scratch/dummy-hybrid/test_results/test_2020_1_2025-06-10_14-04-33_g89c3n70_cleaned.zarr'

ds = xr.open_zarr(path)

ds

/tmp/ipykernel_9976/3146492389.py:5: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_zarr(path)


<xarray.Dataset> Size: 31GB
Dimensions:               (prediction_timedelta: 1, time: 1460, level: 13,
                           longitude: 240, latitude: 121)
Coordinates:
  * prediction_timedelta  (prediction_timedelta) timedelta64[ns] 8B 06:00:00
  * time                  (time) datetime64[ns] 12kB 2020-01-01 ... 2020-12-3...
  * level                 (level) int64 104B 50 100 150 200 ... 700 850 925 1000
  * longitude             (longitude) float64 2kB 0.0 1.5 3.0 ... 357.0 358.5
  * latitude              (latitude) float64 968B -90.0 -88.5 ... 88.5 90.0
Data variables:
    temperature           (prediction_timedelta, time, level, longitude, latitude) float64 4GB dask.array<chunksize=(1, 2, 7, 120, 61), meta=np.ndarray>
    u_component_of_wind   (prediction_timedelta, time, level, longitude, latitude) float64 4GB dask.array<chunksize=(1, 2, 7, 120, 61), meta=np.ndarray>
    vorticity             (prediction_timedelta, time, level, longitude, latitude) float64 4GB dask.array<chunksize=(1, 2, 7, 120, 61), meta=np.ndarray>
    specific_humidity     (prediction_timedelta, time, level, longitude, latitude) float64 4GB dask.array<chunksize=(1, 2, 7, 120, 61), meta=np.ndarray>
    divergence            (prediction_timedelta, time, level, longitude, latitude) float64 4GB dask.array<chunksize=(1, 2, 7, 120, 61), meta=np.ndarray>
    v_component_of_wind   (prediction_timedelta, time, level, longitude, latitude) float64 4GB dask.array<chunksize=(1, 2, 7, 120, 61), meta=np.ndarray>
    geopotential          (prediction_timedelta, time, level, longitude, latitude) float64 4GB dask.array<chunksize=(1, 2, 7, 120, 61), meta=np.ndarray>

In [2]:
path = '/itet-stor/bszarvas/net_scratch/dummy-hybrid/test_results/test_2020_4_2025-06-10_10-33-38_5zsh5vt2.zarr'

def copy_zarr_dataset(
    zarr_path: str,
    suffix: str = '_COPY'
) -> str:
    """
    Read an xarray Zarr dataset from zarr_path, make a copy under a new path with suffix,
    and return the new path.

    Args:
      zarr_path: Path to the source Zarr dataset.
      suffix: Suffix to append to the base name for the copy.
    Returns:
      The path to the newly created Zarr copy.
    """
    import xarray as xr
    import os

    # Open the original Zarr dataset without consolidated metadata
    ds = xr.open_zarr(zarr_path, consolidated=False)
    # Determine the new path based on the suffix
    dir_name, base = os.path.split(zarr_path.rstrip('/'))
    if base.endswith('.zarr'):
        name = base[:-5]
        new_base = f"{name}{suffix}.zarr"
    else:
        new_base = base + suffix
    new_path = os.path.join(dir_name, new_base)
    # Save the dataset to the new path with consolidation
    ds.to_zarr(new_path, mode='w', consolidated=True, zarr_format=2)
    return new_path

new_path = copy_zarr_dataset(path)

print(new_path)


/tmp/ipykernel_9976/1547977094.py:21: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_zarr(zarr_path, consolidated=False)


/itet-stor/bszarvas/net_scratch/dummy-hybrid/test_results/test_2020_4_2025-06-10_10-33-38_5zsh5vt2_COPY.zarr


In [8]:
import xarray as xr
import numpy as np

# 1) Open your Zarr store
ds = xr.open_zarr('/itet-stor/bszarvas/net_scratch/dummy-hybrid/test_results/test_2020_4_2025-06-10_12-33-00_7kr4kf0b_cleaned.zarr')

# 2) For each data variable, count NaNs and total entries
overall_nans = 0
overall_total = 0

for var, da in ds.data_vars.items():
    # this returns a NumPy scalar, not a Dask array
    n_nan = int(da.isnull().sum().values)
    total = da.size
    print(f"{var}: {n_nan}/{total}")

nan_counts = ds.isnull().sum().compute()

for var, da in nan_counts.data_vars.items():
    print(f"NaNS in {var}: {int(da.item())} out of {ds[var].size} NaNs")

/tmp/ipykernel_9976/3207032298.py:5: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_zarr('/itet-stor/bszarvas/net_scratch/dummy-hybrid/test_results/test_2020_4_2025-06-10_12-33-00_7kr4kf0b_cleaned.zarr')


u_component_of_wind: 30404993/552689280
vorticity: 30404993/552689280
temperature: 30404993/552689280
specific_humidity: 30404993/552689280
v_component_of_wind: 30404993/552689280
divergence: 30404993/552689280
geopotential: 30404993/552689280
NaNS in u_component_of_wind: 30404993 out of 552689280 NaNs
NaNS in vorticity: 30404993 out of 552689280 NaNs
NaNS in temperature: 30404993 out of 552689280 NaNs
NaNS in specific_humidity: 30404993 out of 552689280 NaNs
NaNS in v_component_of_wind: 30404993 out of 552689280 NaNs
NaNS in divergence: 30404993 out of 552689280 NaNs
NaNS in geopotential: 30404993 out of 552689280 NaNs


In [13]:
import xarray as xr

path = "/itet-stor/bszarvas/net_scratch/dummy-hybrid/weatherbench_results/results_2020_48h_1_rolled_1.nc"
ds = xr.open_dataset(path)

# 1) Quick overview
print(ds)              # dims, coords, data_vars summary

# 2) List what metrics you have
print("Metrics:", list(ds.data_vars))

# 3) Inspect one metric
#print(ds["rmse"])      # or ds["mse"], ds["bias"], whatever variables you defined

for metric in ds.data_vars:
    print(metric)
    print(ds[metric].values)
    print(ds[metric].values.shape)

<xarray.Dataset> Size: 708B
Dimensions:                   (level: 3, region: 2, lead_time: 1)
Coordinates:
  * level                     (level) int32 12B 500 700 850
  * region                    (region) object 16B 'global' 'northern-hemisphere'
  * lead_time                 (lead_time) timedelta64[ns] 8B 06:00:00
Data variables: (12/14)
    rmse.vorticity            (lead_time, level, region) float64 48B ...
    rmse.divergence           (lead_time, level, region) float64 48B ...
    rmse.v_component_of_wind  (lead_time, level, region) float64 48B ...
    rmse.geopotential         (lead_time, level, region) float64 48B ...
    rmse.specific_humidity    (lead_time, level, region) float64 48B ...
    rmse.temperature          (lead_time, level, region) float64 48B ...
    ...                        ...
    mse.divergence            (lead_time, level, region) float64 48B ...
    mse.v_component_of_wind   (lead_time, level, region) float64 48B ...
    mse.geopotential          (lead_tim

/tmp/ipykernel_9976/1983854611.py:4: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_dataset(path)


In [12]:
import xarray as xr

path = "/itet-stor/bszarvas/net_scratch/dummy-hybrid/weatherbench_results/results_2020_48h_1_not_rolled_1.nc"
ds = xr.open_dataset(path)

# 1) Quick overview
print(ds)              # dims, coords, data_vars summary

# 2) List what metrics you have
print("Metrics:", list(ds.data_vars))

# 3) Inspect one metric
#print(ds["rmse"])      # or ds["mse"], ds["bias"], whatever variables you defined

for metric in ds.data_vars:
    print(metric)
    print(ds[metric].values)
    print(ds[metric].values.shape)


<xarray.Dataset> Size: 708B
Dimensions:                   (level: 3, region: 2, lead_time: 1)
Coordinates:
  * level                     (level) int32 12B 500 700 850
  * region                    (region) object 16B 'global' 'northern-hemisphere'
  * lead_time                 (lead_time) timedelta64[ns] 8B 06:00:00
Data variables: (12/14)
    rmse.u_component_of_wind  (lead_time, level, region) float64 48B ...
    rmse.temperature          (lead_time, level, region) float64 48B ...
    rmse.geopotential         (lead_time, level, region) float64 48B ...
    rmse.v_component_of_wind  (lead_time, level, region) float64 48B ...
    rmse.divergence           (lead_time, level, region) float64 48B ...
    rmse.specific_humidity    (lead_time, level, region) float64 48B ...
    ...                        ...
    mse.temperature           (lead_time, level, region) float64 48B ...
    mse.geopotential          (lead_time, level, region) float64 48B ...
    mse.v_component_of_wind   (lead_tim

/tmp/ipykernel_9976/2297160196.py:4: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_dataset(path)


In [ ]:
import xarray as xr
path_weatherbench2 = "gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr"
ds_bench2 = xr.open_zarr(path_weatherbench2)






<xarray.Dataset> Size: 2TB
Dimensions:                                           (time: 93544,
                                                       longitude: 240,
                                                       latitude: 121, level: 13)
Coordinates:
  * latitude                                          (latitude) float64 968B ...
  * level                                             (level) int64 104B 50 ....
  * longitude                                         (longitude) float64 2kB ...
  * time                                              (time) datetime64[ns] 748kB ...
Data variables: (12/62)
    10m_u_component_of_wind                           (time, longitude, latitude) float32 11GB dask.array<chunksize=(8, 240, 121), meta=np.ndarray>
    10m_v_component_of_wind                           (time, longitude, latitude) float32 11GB dask.array<chunksize=(8, 240, 121), meta=np.ndarray>
    10m_wind_speed                                    (time, longitude, latitude) float32 11GB dask.array<chunksize=(8, 240, 121), meta=np.ndarray>
    2m_dewpoint_temperature                           (time, longitude, latitude) float32 11GB dask.array<chunksize=(8, 240, 121), meta=np.ndarray>
    2m_temperature                                    (time, longitude, latitude) float32 11GB dask.array<chunksize=(8, 240, 121), meta=np.ndarray>
    above_ground                                      (time, level, longitude, latitude) float32 141GB dask.array<chunksize=(8, 13, 240, 121), meta=np.ndarray>
    ...                                                ...
    volumetric_soil_water_layer_1                     (time, longitude, latitude) float32 11GB dask.array<chunksize=(8, 240, 121), meta=np.ndarray>
    volumetric_soil_water_layer_2                     (time, longitude, latitude) float32 11GB dask.array<chunksize=(8, 240, 121), meta=np.ndarray>
    volumetric_soil_water_layer_3                     (time, longitude, latitude) float32 11GB dask.array<chunksize=(8, 240, 121), meta=np.ndarray>
    volumetric_soil_water_layer_4                     (time, longitude, latitude) float32 11GB dask.array<chunksize=(8, 240, 121), meta=np.ndarray>
    vorticity                                         (time, level, longitude, latitude) float32 141GB dask.array<chunksize=(8, 13, 240, 121), meta=np.ndarray>
    wind_speed                                        (time, level, longitude, latitude) float32 141GB dask.array<chunksize=(8, 13, 240, 121), meta=np.ndarray>

In [1]:
## FUNCTION THAT ADDS A NEW AXIS NAMED "PREDICTION_TIMEDELTA"

import xarray as xr
import numpy as np
FORECAST_PATH = '/itet-stor/bszarvas/net_scratch/dummy-hybrid/test_results/test_2020_8_2025-06-03_14-03-32_dzgynvfu_COPY.zarr'
# open your cleaned forecast store
ds = xr.open_zarr(FORECAST_PATH, consolidated=False)

# suppose your model was run for a single horizon of 48 h:
delta = np.timedelta64(8 * 6, 'h')   # e.g. prediction_range=8 ⇒ 48 h

# add that as a new axis named exactly "prediction_timedelta"
ds = ds.expand_dims({'prediction_timedelta': [delta]})

# overwrite the Zarr store (or write to a new directory)
ds.to_zarr(FORECAST_PATH, mode='w', consolidated=True, zarr_format=2)

/itet-stor/bszarvas/net_scratch/conda_envs/nergcm/lib/python3.13/site-packages/zarr/core/group.py:3301: UserWarning: Object at gmon.out is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


In [16]:
### FUNCTION THAT SHIFTS TIME INDEX BY 48 HOURS

import xarray as xr
import numpy as np

# 1) point this at your store (Zarr or .nc)
path = '/itet-stor/bszarvas/net_scratch/dummy-hybrid/test_results/test_2020_8_2025-06-03_14-03-32_dzgynvfu'    # or '…/dataset.nc'

# 2) open it lazily
ds = xr.open_zarr(path, consolidated=False)  # if netCDF: xr.open_dataset(path)

# 3) shift the time index by 48h
delta = np.timedelta64(6*8, 'h')
ds = ds.assign_coords(time=ds['time'] + delta)

# 4) write back, clobbering the old one
ds.to_zarr(path, mode='w', consolidated=True, zarr_format=2)   # if netCDF: ds.to_netcdf(path)


In [11]:
import xarray as xr
import numpy as np

# 1) Open your Zarr store
ds = xr.open_zarr('/itet-stor/bszarvas/net_scratch/dummy-hybrid/test_results/test_2020_4_2025-06-04_15-03-23_dpgmur2r_COPY.zarr')

# 2) For each data variable, count NaNs and total entries
overall_nans = 0
overall_total = 0

for var, da in ds.data_vars.items():
    # this returns a NumPy scalar, not a Dask array
    n_nan = int(da.isnull().sum().values)
    total = da.size
    print(f"{var}: {n_nan}/{total}")

# 3) Print an overall tally across *all* variables
print()
print(f"Overall: {overall_nans} NaNs out of {overall_total} total values "
      f"({100 * overall_nans/overall_total:.2f}%)")

/tmp/ipykernel_202058/1779038568.py:5: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_zarr('/itet-stor/bszarvas/net_scratch/dummy-hybrid/test_results/test_2020_4_2025-06-04_15-03-23_dpgmur2r_COPY.zarr')


temperature: 552689280/552689280
u_component_of_wind: 552689280/552689280
vorticity: 552689280/552689280
specific_humidity: 552689280/552689280
v_component_of_wind: 552689280/552689280
geopotential: 552689280/552689280
divergence: 552689280/552689280



ZeroDivisionError: division by zero

In [ ]:
import xarray as xr
import numpy as np

# 1) Open your Zarr store
ds = xr.open_zarr('/itet-stor/bszarvas/net_scratch/dummy-hybrid/test_results/test_2020_4_2025-06-04_16-48-32_5juamwul.zarr')

# 2) For each data variable, count NaNs and total entries
overall_nans = 0
overall_total = 0

for var, da in ds.data_vars.items():
    # this returns a NumPy scalar, not a Dask array
    n_nan = int(da.isnull().sum().values)
    total = da.size
    print(f"{var}: {n_nan}/{total}")

# 3) Print an overall tally across *all* variables
print()
print(f"Overall: {overall_nans} NaNs out of {overall_total} total values "
      f"({100 * overall_nans/overall_total:.2f}%)")

temperature: 108194/1887600
u_component_of_wind: 108194/1887600
specific_humidity: 108194/1887600
vorticity: 108194/1887600
v_component_of_wind: 108194/1887600
divergence: 108194/1887600
geopotential: 108194/1887600



ZeroDivisionError: division by zero

In [ ]:
import xarray as xr
import numpy as np

# 1) Open your Zarr store
ds = xr.open_zarr('test_results/test_2020_4_2025-06-04_09-00-46_o07ya0il.zarr')

# 2) For each data variable, count NaNs and total entries
overall_nans = 0
overall_total = 0

for var, da in ds.data_vars.items():
    # number of NaNs
    n_nan = int(da.isnull().sum().item())
    # total number of grid‐points in that variable
    total = da.size
    overall_nans += n_nan
    overall_total += total

    print(f"{var}: {n_nan} NaNs out of {total} entries ({100 * n_nan/total:.2f}%)")

# 3) Print an overall tally across *all* variables
print()
print(f"Overall: {overall_nans} NaNs out of {overall_total} total values "
      f"({100 * overall_nans/overall_total:.2f}%)")

/tmp/ipykernel_244693/337208676.py:5: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_zarr('test_results/test_2020_4_2025-06-04_09-00-46_o07ya0il.zarr')


NotImplementedError: 'item' is not yet a valid method on dask arrays

In [9]:
import xarray as xr

ds = xr.open_zarr('/itet-stor/bszarvas/net_scratch/dummy-hybrid/test_results/test_2020_4_2025-06-10_12-33-00_7kr4kf0b_cleaned.zarr')

ds

/tmp/ipykernel_9976/2857314711.py:3: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_zarr('/itet-stor/bszarvas/net_scratch/dummy-hybrid/test_results/test_2020_4_2025-06-10_12-33-00_7kr4kf0b_cleaned.zarr')


<xarray.Dataset> Size: 31GB
Dimensions:               (prediction_timedelta: 1, time: 1464, level: 13,
                           longitude: 240, latitude: 121)
Coordinates:
  * prediction_timedelta  (prediction_timedelta) timedelta64[ns] 8B 1 days
  * time                  (time) datetime64[ns] 12kB 2020-01-01 ... 2020-12-3...
  * longitude             (longitude) float64 2kB 0.0 1.5 3.0 ... 357.0 358.5
  * latitude              (latitude) float64 968B -90.0 -88.5 ... 88.5 90.0
  * level                 (level) int64 104B 50 100 150 200 ... 700 850 925 1000
Data variables:
    u_component_of_wind   (prediction_timedelta, time, level, longitude, latitude) float64 4GB dask.array<chunksize=(1, 2, 7, 120, 61), meta=np.ndarray>
    vorticity             (prediction_timedelta, time, level, longitude, latitude) float64 4GB dask.array<chunksize=(1, 2, 7, 120, 61), meta=np.ndarray>
    temperature           (prediction_timedelta, time, level, longitude, latitude) float64 4GB dask.array<chunksize=(1, 2, 7, 120, 61), meta=np.ndarray>
    specific_humidity     (prediction_timedelta, time, level, longitude, latitude) float64 4GB dask.array<chunksize=(1, 2, 7, 120, 61), meta=np.ndarray>
    v_component_of_wind   (prediction_timedelta, time, level, longitude, latitude) float64 4GB dask.array<chunksize=(1, 2, 7, 120, 61), meta=np.ndarray>
    divergence            (prediction_timedelta, time, level, longitude, latitude) float64 4GB dask.array<chunksize=(1, 2, 7, 120, 61), meta=np.ndarray>
    geopotential          (prediction_timedelta, time, level, longitude, latitude) float64 4GB dask.array<chunksize=(1, 2, 7, 120, 61), meta=np.ndarray>

In [ ]:
from dinosaur import coordinate_systems, filtering, primitive_equations, spherical_harmonic, sigma_coordinates, scales, vertical_interpolation
from utils import xarray_interpolate
import time
import xarray as xr

def setup_pressure_coordinates(physics_specs, ds):
    variables = ['u_component_of_wind']
    ds = ds[variables] 

    pressure_levels = ds.level.values
    nondim_pressure_centers = physics_specs.nondimensionalize(
        pressure_levels * scales.units.millibar
    )
    return vertical_interpolation.PressureCoordinates(nondim_pressure_centers)

def setup_target_coordinates(vertical_grid):
    """Set up source coordinate system based on the dataset.
    
    Args:
        ds: Dataset on equiangular grid with pressure levels
    """
    lons = 240
    
    # Calculate appropriate parameters for Grid.construct
    max_wavenumber = int(lons / 3) - 1

    # Create equiangular grid using the construct method
    horizontal_grid = spherical_harmonic.Grid(
        longitude_nodes=240,
        latitude_nodes=121,
        latitude_spacing='equiangular_with_poles',  # Required for odd number of nodes
        longitude_wavenumbers=max_wavenumber,        # Choose appropriate spectral resolution 
        total_wavenumbers=max_wavenumber,
    )
    
    # Create source coordinate system
    return coordinate_systems.CoordinateSystem(horizontal_grid, vertical_grid)


ds_reference = xr.open_zarr("gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr")

physics_specs = primitive_equations.PrimitiveEquationsSpecs.from_si()

pressure_coords = setup_pressure_coordinates(physics_specs, ds_reference)

target_grid = setup_target_coordinates(pressure_coords)

ds_path = '/itet-stor/bszarvas/net_scratch/dummy-hybrid/test_results/test_2020_6_5htjcdbk'

start_time = time.time()
xarray_interpolate(ds_path, target_grid)
end_time = time.time()
print(f"Interpolation took {end_time - start_time} seconds")





Interpolation took 1.4542269706726074 seconds


/itet-stor/bszarvas/net_scratch/conda_envs/nergcm/lib/python3.13/site-packages/zarr/api/asynchronous.py:203: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [8]:
%%time
# ---------------------------------------------------------------------------
# 0.  Imports and paths  (unchanged from your script)
# ---------------------------------------------------------------------------
from dinosaur import (
    primitive_equations, scales
)                                       # same modules you already use
from utils import xarray_interpolate     # if you need it later
import xarray as xr
import dask                              # dask is implicit in xarray
import os

# overwrite these two strings as needed
PATH_IN   = '/itet-stor/bszarvas/net_scratch/dummy-hybrid/test_results/test_2020_6_2025-05-31_16-59-21_dwk8dx1a'
PATH_OUT   = '/itet-stor/bszarvas/net_scratch/dummy-hybrid/test_results/test_2020_6_2025-05-31_16-59-21_dwk8dx1a_MAYBEGOOD'
PATH_ERA5  = 'gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr'

# ---------------------------------------------------------------------------
# 1.  Load datasets (unchanged)
# ---------------------------------------------------------------------------
physics_specs = primitive_equations.PrimitiveEquationsSpecs.from_si()

ds_target     = xr.open_zarr(PATH_IN)        # local target grid
ds_reference  = xr.open_zarr(PATH_ERA5)       # remote ERA-5 archive

ds_filtered   = ds_reference[ds_target.keys()]    # keep matching vars only

# ---------------------------------------------------------------------------
# 2.  Attach physical units (unchanged helper)
# ---------------------------------------------------------------------------
def attach_data_array_units(arr):
    attrs = dict(arr.attrs)
    units = attrs.pop("units", None)
    if units in {"(0-1)", "%", "~"}:
        units = None
    q = scales.units.parse_expression(units) if units else scales.units.dimensionless
    return xr.DataArray(q * arr.data, arr.coords, arr.dims, attrs=attrs)

def attach_xarray_units(ds):
    return ds.map(attach_data_array_units)

ds_filtered = attach_xarray_units(ds_filtered)

# unit lookup, plus three explicit overrides just like before
units_dict = {k: v.data.units for k, v in ds_filtered.items()}
units_dict |= {
    "vorticity":   scales.units("1/s"),
    "divergence":  scales.units("1/s"),
    "geopotential": scales.units("m^2/s^2"),
}

# ---------------------------------------------------------------------------
# 3.  **Fast** dimensionalisation  (replaces the slow apply_ufunc loop)
# ---------------------------------------------------------------------------
def xarray_dimensionalize_fast(ds, physics_specs, unit_map):
    """Vectorised multiply per variable, return *numeric* dask arrays."""
    result = {}
    for name, da in ds.items():
        beta = physics_specs.dimensionalize(1.0, unit_map[name]).magnitude
        scaled = da * beta                 # pint.Quantity * dask → Quantity
        if hasattr(scaled, "magnitude"):   # strip pint wrapper
            scaled = scaled.magnitude      # -> dask.array.Array (numeric)
        result[name] = xr.DataArray(
            scaled,
            coords=da.coords,
            dims=da.dims,
            attrs=da.attrs,
        )
    return xr.Dataset(result, coords=ds.coords, attrs=ds.attrs)

print("Everything else is successful before xarray_dimensionalize_fast")

ds_si = xarray_dimensionalize_fast(ds_target, physics_specs, units_dict)

print("xarray_dimensionalize_fast is successful")

def strip_units_from_coords(ds):
    """Return a copy of *ds* whose coordinates are plain ndarrays."""
    return ds.assign_coords(
        {c: (ds[c].dims, ds[c].data) for c in ds.coords}
    )

ds_si_clean = strip_units_from_coords(ds_si)

ds_si = ds_si.chunk()

print("ds_si is chunked")

# ---------------------------------------------------------------------------
# 4.  Store result (unchanged)
# ---------------------------------------------------------------------------
ds_si.to_zarr(PATH_OUT, mode="w", consolidated=True, zarr_format=2)
print("✓ Saved SI dataset to", PATH_OUT)



Everything else is successful before xarray_dimensionalize_fast
xarray_dimensionalize_fast is successful
ds_si is chunked
✓ Saved SI dataset to /itet-stor/bszarvas/net_scratch/dummy-hybrid/test_results/test_2020_6_2025-05-31_16-59-21_dwk8dx1a_MAYBEGOOD
CPU times: user 483 ms, sys: 95.5 ms, total: 578 ms
Wall time: 1.35 s


In [3]:
import xarray as xr

path = '/itet-stor/bszarvas/net_scratch/dummy-hybrid/test_results/test_2020_6_2025-05-31_14-37-30_hcmgk567'

ds = xr.open_zarr(PATH_OUT)

ds



<xarray.Dataset> Size: 169MB
Dimensions:              (time: 8, level: 13, lon: 240, lat: 121)
Coordinates:
  * time                 (time) datetime64[ns] 64B 2020-01-01 ... 2020-01-02T...
  * lat                  (lat) float64 968B -90.0 -88.5 -87.0 ... 87.0 88.5 90.0
  * level                (level) float32 52B 0.03846 0.1154 ... 0.8846 0.9615
  * lon                  (lon) float64 2kB 0.0 1.5 3.0 4.5 ... 355.5 357.0 358.5
Data variables:
    temperature          (time, level, lon, lat) float64 24MB dask.array<chunksize=(2, 7, 120, 61), meta=np.ndarray>
    u_component_of_wind  (time, level, lon, lat) float64 24MB dask.array<chunksize=(2, 7, 120, 61), meta=np.ndarray>
    vorticity            (time, level, lon, lat) float64 24MB dask.array<chunksize=(2, 7, 120, 61), meta=np.ndarray>
    specific_humidity    (time, level, lon, lat) float64 24MB dask.array<chunksize=(2, 7, 120, 61), meta=np.ndarray>
    v_component_of_wind  (time, level, lon, lat) float64 24MB dask.array<chunksize=(2, 7, 120, 61), meta=np.ndarray>
    divergence           (time, level, lon, lat) float64 24MB dask.array<chunksize=(2, 7, 120, 61), meta=np.ndarray>
    geopotential         (time, level, lon, lat) float64 24MB dask.array<chunksize=(2, 7, 120, 61), meta=np.ndarray>
Attributes:
    longitude_wavenumbers:     128
    total_wavenumbers:         129
    longitude_nodes:           256
    latitude_nodes:            128
    latitude_spacing:          gauss
    longitude_offset:          0.0
    radius:                    1.0
    spherical_harmonics_impl:  RealSphericalHarmonics
    spmd_mesh:                 
    boundaries:                [0.0, 0.07692307978868484, 0.1538461595773697,...
    horizontal_grid_type:      Grid
    vertical_grid_type:        SigmaCoordinates

In [7]:
%%time
from dinosaur import coordinate_systems, filtering, primitive_equations, spherical_harmonic, sigma_coordinates, scales, vertical_interpolation
from utils import xarray_interpolate
import time
import xarray as xr

import xarray as xr

path = '/itet-stor/bszarvas/net_scratch/dummy-hybrid/test_results/test_2020_6_2025-05-31_16-59-21_dwk8dx1a'

out_path = '/itet-stor/bszarvas/net_scratch/dummy-hybrid/test_results/test_2020_6_2025-05-31_16-59-21_dwk8dx1a_MAYBEGOOD_2'

def xarray_dimensionalize(ds: xr.Dataset,
                          physics_specs,
                          unit_map: dict):
  """
  Take a nondimensional xarray.Dataset and return a new Dataset whose
  data_vars have been converted back to SI via physics_specs.dimensionalize.
  
  Args:
    ds:       xarray.Dataset of pure floats (model nondimensional units)
    physics_specs: your Dinosaur PrimitiveEquationsSpecs
    unit_map: dict mapping each ds.data_vars key → a pint.Unit
  
  Returns:
    new xarray.Dataset of floats in SI
  """
  def _dim_func(x, unit):
    # x is a numpy/jax scalar or array; returns a plain numpy array
    return physics_specs.dimensionalize(x, unit).magnitude

  out = xr.Dataset(coords=ds.coords, attrs=ds.attrs)
  for var in ds.data_vars:
    u = unit_map[var]
    da = ds[var]
    out[var] = xr.apply_ufunc(
      _dim_func,
      da,
      input_core_dims=[[]],       # operate element‐wise
      kwargs={"unit": u},
      vectorize=True,             # wrap in jnp.vectorize
      dask="parallelized",        # if you happen to have chunks
      output_dtypes=[float],      # result is a float array
    )
  return out

def attach_data_array_units(array):
  attrs = dict(array.attrs)
  units = attrs.pop('units', None)
  if units in {'(0-1)', '%', '~'}:
    units = None
  if units is not None:
    data = scales.units.parse_expression(units) * array.data
  else:
    data = scales.units.dimensionless * array.data
  return xr.DataArray(data, array.coords, array.dims, attrs=attrs)

def attach_xarray_units(ds):
  return ds.map(attach_data_array_units)


physics_specs = primitive_equations.PrimitiveEquationsSpecs.from_si()

ds_target = xr.open_zarr(path)


ds_reference = xr.open_zarr("gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr")

print(ds_target.keys())

ds_filtered = ds_reference[ds_target.keys()]

ds_filtered = attach_xarray_units(ds_filtered)

units_dict = {k: v.data.units for k,v in ds_filtered.items()}

units_dict |= {'vorticity': scales.units('1/s'), 'divergence': scales.units('1/s'), 'geopotential': scales.units('m^2/s^2')}

print(units_dict)


ds_si = xarray_dimensionalize(ds_target, physics_specs, units_dict)

print("nondimensionalization is successful")

ds_si.to_zarr(path, mode='w', consolidated=True)



<timed exec>:63: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.


KeysView(<xarray.Dataset> Size: 48MB
Dimensions:            (time: 8, level: 13, lon: 240, lat: 121)
Dimensions without coordinates: time, level, lon, lat
Data variables:
    divergence         (time, level, lon, lat) float64 24MB dask.array<chunksize=(2, 7, 120, 61), meta=np.ndarray>
    specific_humidity  (time, level, lon, lat) float64 24MB dask.array<chunksize=(2, 7, 120, 61), meta=np.ndarray>
Attributes:
    longitude_wavenumbers:     128
    total_wavenumbers:         129
    longitude_nodes:           256
    latitude_nodes:            128
    latitude_spacing:          gauss
    longitude_offset:          0.0
    radius:                    1.0
    spherical_harmonics_impl:  RealSphericalHarmonics
    spmd_mesh:                 
    boundaries:                [0.0, 0.07692307978868484, 0.1538461595773697,...
    horizontal_grid_type:      Grid
    vertical_grid_type:        SigmaCoordinates)
{'divergence': <Quantity(1.0, '1 / second')>, 'specific_humidity': <Unit('dimensionless'

/itet-stor/bszarvas/net_scratch/conda_envs/nergcm/lib/python3.13/site-packages/zarr/api/asynchronous.py:203: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


KeyboardInterrupt: 

Interpolation took 1.4542269706726074 seconds


/itet-stor/bszarvas/net_scratch/conda_envs/nergcm/lib/python3.13/site-packages/zarr/api/asynchronous.py:203: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [ ]:
interpolated = xr.open_zarr(ds_path)

interpolated

<xarray.Dataset> Size: 85MB
Dimensions:              (level: 13, time: 4, lon: 240, lat: 121)
Coordinates:
  * level                (level) float32 52B 0.03846 0.1154 ... 0.8846 0.9615
  * time                 (time) datetime64[ns] 32B 2019-12-30T12:00:00 ... 20...
  * lat                  (lat) float64 968B -90.0 -88.5 -87.0 ... 87.0 88.5 90.0
  * lon                  (lon) float64 2kB 0.0 1.5 3.0 4.5 ... 355.5 357.0 358.5
Data variables:
    vorticity            (time, level, lon, lat) float64 12MB dask.array<chunksize=(2, 7, 120, 61), meta=np.ndarray>
    u_component_of_wind  (time, level, lon, lat) float64 12MB dask.array<chunksize=(2, 7, 120, 61), meta=np.ndarray>
    temperature          (time, level, lon, lat) float64 12MB dask.array<chunksize=(2, 7, 120, 61), meta=np.ndarray>
    divergence           (time, level, lon, lat) float64 12MB dask.array<chunksize=(2, 7, 120, 61), meta=np.ndarray>
    specific_humidity    (time, level, lon, lat) float64 12MB dask.array<chunksize=(2, 7, 120, 61), meta=np.ndarray>
    v_component_of_wind  (time, level, lon, lat) float64 12MB dask.array<chunksize=(2, 7, 120, 61), meta=np.ndarray>
    geopotential         (time, level, lon, lat) float64 12MB dask.array<chunksize=(2, 7, 120, 61), meta=np.ndarray>
Attributes:
    longitude_wavenumbers:     128
    total_wavenumbers:         129
    longitude_nodes:           256
    latitude_nodes:            128
    latitude_spacing:          gauss
    longitude_offset:          0.0
    radius:                    1.0
    spherical_harmonics_impl:  RealSphericalHarmonics
    spmd_mesh:                 
    boundaries:                [0.0, 0.07692307978868484, 0.1538461595773697,...
    horizontal_grid_type:      Grid
    vertical_grid_type:        SigmaCoordinates

In [10]:
import jax.numpy as jnp

vector = jnp.arange(5)

vector2 = jnp.arange(10, 20)

print(vector)
print(vector2)

grid1, grid2 = jnp.meshgrid(vector, vector2, indexing='ij')

grid1 = grid1.flatten()

grid2 = grid2.flatten()

print(grid1)
print(grid2)



[0 1 2 3 4]
[10 11 12 13 14 15 16 17 18 19]
[0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 3 3 3 3 3 3 3
 3 3 3 4 4 4 4 4 4 4 4 4 4]
[10 11 12 13 14 15 16 17 18 19 10 11 12 13 14 15 16 17 18 19 10 11 12 13
 14 15 16 17 18 19 10 11 12 13 14 15 16 17 18 19 10 11 12 13 14 15 16 17
 18 19]


In [12]:
import jax
import jax.numpy as jnp

def construct_graph_jit_compatible(coords):
    """Graph construction that works with JIT."""
    # Get grid dimensions
    lon_size, lat_size = coords
    num_nodes = lon_size * lat_size
    max_edges = num_nodes * 9  # Maximum possible edges (9-point stencil)
    
    # Create meshgrid
    i_indices, j_indices = jnp.meshgrid(jnp.arange(lon_size), jnp.arange(lat_size), indexing='ij')
    i_indices = i_indices.reshape(-1)
    j_indices = j_indices.reshape(-1)
    
    # Define offsets for 9-point stencil
    di_offsets = jnp.array([-1, -1, -1, 0, 0, 0, 1, 1, 1])
    dj_offsets = jnp.array([-1, 0, 1, -1, 0, 1, -1, 0, 1])
    
    # Pre-allocate arrays of fixed size
    senders = jnp.zeros(max_edges, dtype=jnp.int32)
    receivers = jnp.zeros(max_edges, dtype=jnp.int32)
    valid = jnp.zeros(max_edges, dtype=jnp.bool_)
    
    # Function to compute linear index
    def linear_index(i, j):
        return i * lat_size + j
    
    # Process all nodes and stencil offsets
    def process_node(idx, arrays):
        senders, receivers, valid = arrays
        node_i, node_j = i_indices[idx], j_indices[idx]
        node_idx = linear_index(node_i, node_j)
        
        # Base offset for this node's edges
        base_offset = idx * 9
        
        # Process all 9 stencil points
        for k in range(9):
            ni = (node_i + di_offsets[k]) % lon_size  # Apply longitude periodicity
            nj = node_j + dj_offsets[k]
            
            # Check if neighbor is valid
            is_valid = (nj >= 0) & (nj < lat_size)
            
            # Calculate edge index
            edge_idx = base_offset + k
            
            # Set values using dynamic_update_slice
            senders = senders.at[edge_idx].set(
                jnp.where(is_valid, node_idx, 0))
            receivers = receivers.at[edge_idx].set(
                jnp.where(is_valid, linear_index(ni, nj), 0))
            valid = valid.at[edge_idx].set(is_valid)
            
        return senders, receivers, valid
            
    # Scan over all nodes
    init_arrays = (senders, receivers, valid)
    final_arrays = jax.lax.fori_loop(0, num_nodes, process_node, init_arrays)
    
    # Get final arrays
    senders, receivers, valid = final_arrays
    
    # Count actual valid edges
    num_valid = jnp.sum(valid)
    
    # Outputs that work with GraphsTuple
    return senders, receivers, num_nodes, num_valid, valid

senders, recievers, num_nodes, num_valid, valid = construct_graph_jit_compatible((5, 7))

print(senders)
print(recievers)
print(num_nodes)
print(num_valid)
print(valid)

[ 0  0  0  0  0  0  0  0  0  1  1  1  1  1  1  1  1  1  2  2  2  2  2  2
  2  2  2  3  3  3  3  3  3  3  3  3  4  4  4  4  4  4  4  4  4  5  5  5
  5  5  5  5  5  5  6  6  0  6  6  0  6  6  0  0  7  7  0  7  7  0  7  7
  8  8  8  8  8  8  8  8  8  9  9  9  9  9  9  9  9  9 10 10 10 10 10 10
 10 10 10 11 11 11 11 11 11 11 11 11 12 12 12 12 12 12 12 12 12 13 13  0
 13 13  0 13 13  0  0 14 14  0 14 14  0 14 14 15 15 15 15 15 15 15 15 15
 16 16 16 16 16 16 16 16 16 17 17 17 17 17 17 17 17 17 18 18 18 18 18 18
 18 18 18 19 19 19 19 19 19 19 19 19 20 20  0 20 20  0 20 20  0  0 21 21
  0 21 21  0 21 21 22 22 22 22 22 22 22 22 22 23 23 23 23 23 23 23 23 23
 24 24 24 24 24 24 24 24 24 25 25 25 25 25 25 25 25 25 26 26 26 26 26 26
 26 26 26 27 27  0 27 27  0 27 27  0  0 28 28  0 28 28  0 28 28 29 29 29
 29 29 29 29 29 29 30 30 30 30 30 30 30 30 30 31 31 31 31 31 31 31 31 31
 32 32 32 32 32 32 32 32 32 33 33 33 33 33 33 33 33 33 34 34  0 34 34  0
 34 34  0]
[ 0 28 29  0  0  1  0  7  8 28 29 30  0 

In [1]:
import torch
print('__CUDNN VERSION:', torch.backends.cudnn.version())
print('__Number CUDA Devices:', torch.cuda.device_count())
print('__CUDA Device Name:',torch.cuda.get_device_name(0))
print('__CUDA Device Total Memory [GB]:',torch.cuda.get_device_properties(0).total_memory/1e9)

__CUDNN VERSION: 90100
__Number CUDA Devices: 4
__CUDA Device Name: NVIDIA A100 80GB PCIe
__CUDA Device Total Memory [GB]: 84.98774016


In [1]:
import jax
print(len(jax.devices()))

1


Run data_cacher.py

In [30]:
cmd = """
python /itet-stor/bszarvas/net_scratch/dummy-hybrid/data_cacher.py \
--data_path "gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr" \
--cache_dir "/scratch/bszarvas/cache" \
--start_time "1994-01-01T00:00" \
--end_time "2017-12-31T06:00" \
--resolution TL127 \
--num_levels 13 \
--batch_size 1 \
"""
!$cmd


Vertical grid: SigmaCoordinates(boundaries=array([0.        , 0.07692308, 0.15384616, 0.23076923, 0.30769232,
       0.3846154 , 0.46153846, 0.53846157, 0.61538464, 0.6923077 ,
       0.7692308 , 0.84615386, 0.9230769 , 1.        ], dtype=float32))
Target coordinate system: CoordinateSystem(horizontal=Grid(longitude_wavenumbers=128, total_wavenumbers=129, longitude_nodes=256, latitude_nodes=128, latitude_spacing='gauss', longitude_offset=0.0, radius=1.0, spherical_harmonics_impl=<class 'dinosaur.spherical_harmonic.RealSphericalHarmonics'>, spmd_mesh=None), vertical=SigmaCoordinates(boundaries=array([0.        , 0.07692308, 0.15384616, 0.23076923, 0.30769232,
       0.3846154 , 0.46153846, 0.53846157, 0.61538464, 0.6923077 ,
       0.7692308 , 0.84615386, 0.9230769 , 1.        ], dtype=float32)), spmd_mesh=None, dycore_partition_spec=PartitionSpec('z', 'x', 'y'), physics_partition_spec=PartitionSpec(None, ('x', 'z'), 'y'))
Physics specifications: PrimitiveEquationsSpecs(radius=1.0, angu

Run experiment.py

In [ ]:
cmd = """
python /itet-stor/bszarvas/net_scratch/dummy-hybrid/experiment.py \
--data_path "gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr" \
--save_path "/itet-stor/bszarvas/net_scratch/dummy-hybrid/models" \
--num_epochs 1 \
--batch_size 1 \
--learning_rate 0.001 \
--wandb_project "hybrid-weather" \
--prediction_range 1 \
--prediction_steps 1 \
--cache_dir "/scratch/bszarvas/cache" \
--resolution TL127 \
--num_levels 32 \
--dt 1.0 \
--start_time "1980-01-01T00:00" \
--end_time "2017-12-31T06:00" \
--dfi_timescale 21600.0
"""
!$cmd

args Namespace(data_path='gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr', save_path='/itet-stor/bszarvas/net_scratch/dummy-hybrid/models', num_epochs=1, batch_size=1, learning_rate=0.001, wandb_project='hybrid-weather', prediction_range=1, prediction_steps=1, cache_dir='/scratch/bszarvas/cache', no_wandb=False, resolution='TL127', num_levels=32, dt=1.0, no_correction=False, start_time='1980-01-01T00:00', end_time='2017-12-31T06:00', dfi_timescale=21600.0, apply_dfi=False)
Training model with resolution: TL127
Vertical grid: SigmaCoordinates(boundaries=array([0.     , 0.03125, 0.0625 , 0.09375, 0.125  , 0.15625, 0.1875 ,
       0.21875, 0.25   , 0.28125, 0.3125 , 0.34375, 0.375  , 0.40625,
       0.4375 , 0.46875, 0.5    , 0.53125, 0.5625 , 0.59375, 0.625  ,
       0.65625, 0.6875 , 0.71875, 0.75   , 0.78125, 0.8125 , 0.84375,
       0.875  , 0.90625, 0.9375 , 0.96875, 1.     ], dtype=float32))
Coordinate system: CoordinateSystem(ho

In [6]:
!python /itet-stor/bszarvas/net_scratch/dummy-hybrid/experiment.py --data_path "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3" --resolution TL31 --num_levels 8 --dt 5.0 --num_epochs 2 --batch_size 1 --learning_rate 0.001 --save_path "/itet-stor/bszarvas/net_scratch/dummy-hybrid/models" --start_time "19900501T00" --end_time "19900501T01" --wandb_project "hybrid-weather"

^C


Dataset inspection

In [9]:
ds_merged = xr.merge([ds1.sel(time='19900501T00').drop_dims('level'), ds2.sel(time='19900501T00')])
ds_merged

<xarray.Dataset> Size: 9GB
Dimensions:                                                          (
                                                                      latitude: 721,
                                                                      longitude: 1440,
                                                                      hybrid: 137)
Coordinates:
  * latitude                                                         (latitude) float32 3kB ...
  * longitude                                                        (longitude) float32 6kB ...
    time                                                             datetime64[ns] 8B ...
  * hybrid                                                           (hybrid) float32 548B ...
Data variables: (12/276)
    100m_u_component_of_wind                                         (latitude, longitude) float32 4MB ...
    100m_v_component_of_wind                                         (latitude, longitude) float32 4MB ...
    10m_u_component_of_neutral_wind                                  (latitude, longitude) float32 4MB ...
    10m_u_component_of_wind                                          (latitude, longitude) float32 4MB ...
    10m_v_component_of_neutral_wind                                  (latitude, longitude) float32 4MB ...
    10m_v_component_of_wind                                          (latitude, longitude) float32 4MB ...
    ...                                                               ...
    specific_snow_water_content                                      (hybrid, latitude, longitude) float32 569MB ...
    temperature                                                      (hybrid, latitude, longitude) float32 569MB ...
    u_component_of_wind                                              (hybrid, latitude, longitude) float32 569MB ...
    v_component_of_wind                                              (hybrid, latitude, longitude) float32 569MB ...
    vertical_velocity                                                (hybrid, latitude, longitude) float32 569MB ...
    vorticity                                                        (hybrid, latitude, longitude) float32 569MB ...
Attributes:
    last_updated:           2025-03-11 02:08:39.963416+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2024-12-31
    valid_time_stop_era5t:  2025-03-05

In [1]:

import xarray as xr
def import_data(path):
    return xr.open_zarr(path, chunks=None, storage_options=dict(token='anon'))


path1 = '"gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr"
ds1 = import_data(path1)


<xarray.Dataset> Size: 1PB
Dimensions:                                                          (
                                                                      time: 1323648,
                                                                      latitude: 721,
                                                                      longitude: 1440)
Coordinates:
  * latitude                                                         (latitude) float32 3kB ...
  * longitude                                                        (longitude) float32 6kB ...
  * time                                                             (time) datetime64[ns] 11MB ...
Data variables: (12/262)
    100m_u_component_of_wind                                         (time, latitude, longitude) float32 5TB ...
    100m_v_component_of_wind                                         (time, latitude, longitude) float32 5TB ...
    10m_u_component_of_neutral_wind                                  (time, latitude, longitude) float32 5TB ...
    10m_u_component_of_wind                                          (time, latitude, longitude) float32 5TB ...
    10m_v_component_of_neutral_wind                                  (time, latitude, longitude) float32 5TB ...
    10m_v_component_of_wind                                          (time, latitude, longitude) float32 5TB ...
    ...                                                               ...
    wave_spectral_directional_width_for_swell                        (time, latitude, longitude) float32 5TB ...
    wave_spectral_directional_width_for_wind_waves                   (time, latitude, longitude) float32 5TB ...
    wave_spectral_kurtosis                                           (time, latitude, longitude) float32 5TB ...
    wave_spectral_peakedness                                         (time, latitude, longitude) float32 5TB ...
    wave_spectral_skewness                                           (time, latitude, longitude) float32 5TB ...
    zero_degree_level                                                (time, latitude, longitude) float32 5TB ...
Attributes:
    last_updated:           2025-04-29 02:21:31.997565+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2025-01-31
    valid_time_stop_era5t:  2025-04-23

In [2]:
import xarray as xr

def import_temperature(path: str):
    # 1. Open *only* the "temperature" variable
    ds = xr.open_zarr(
        path,
        group=None,                            # root group
        chunks={"time": 50,                    # e.g. 50 time‐steps per chunk
                "latitude": 121,                # half of 121
                "longitude": 240},             # half of 240
        storage_options={"token": "anon"},
    )
    # 2. Drop everything except temperature
    da_temp = ds["temperature"]               # DataArray with dask backing
    return da_temp

path = "gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr"
da_temp = import_temperature(path)

# 3. Lazy reduction: this builds a dask graph, no data yet in RAM
lazy_max = da_temp.max(dim=["time", "latitude", "longitude"])
lazy_min = da_temp.min(dim=["time", "latitude", "longitude"])

# 4. Compute – dask will load one chunk at a time, track only intermediate maxima
max_val, min_val = lazy_max.compute(), lazy_min.compute()

print(f"Max temperature: {max_val.values:.3f}")
print(f"Min temperature: {min_val.values:.3f}")





/tmp/ipykernel_45279/2109911303.py:5: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 50. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_zarr(


KeyboardInterrupt: 

In [ ]:
path2 = 'gs://gcp-public-data-arco-era5/ar/model-level-1h-0p25deg.zarr-v1'
ds2 = import_data(path2)
ds2

<xarray.Dataset> Size: 11PB
Dimensions:                              (time: 1323648, hybrid: 137,
                                          latitude: 721, longitude: 1440)
Coordinates:
  * hybrid                               (hybrid) float32 548B 1.0 2.0 ... 137.0
  * latitude                             (latitude) float32 3kB 90.0 ... -90.0
  * longitude                            (longitude) float32 6kB 0.0 ... 359.8
  * time                                 (time) datetime64[ns] 11MB 1900-01-0...
Data variables: (12/14)
    divergence                           (time, hybrid, latitude, longitude) float32 753TB ...
    fraction_of_cloud_cover              (time, hybrid, latitude, longitude) float32 753TB ...
    geopotential                         (time, hybrid, latitude, longitude) float32 753TB ...
    ozone_mass_mixing_ratio              (time, hybrid, latitude, longitude) float32 753TB ...
    specific_cloud_ice_water_content     (time, hybrid, latitude, longitude) float32 753TB ...
    specific_cloud_liquid_water_content  (time, hybrid, latitude, longitude) float32 753TB ...
    ...                                   ...
    specific_snow_water_content          (time, hybrid, latitude, longitude) float32 753TB ...
    temperature                          (time, hybrid, latitude, longitude) float32 753TB ...
    u_component_of_wind                  (time, hybrid, latitude, longitude) float32 753TB ...
    v_component_of_wind                  (time, hybrid, latitude, longitude) float32 753TB ...
    vertical_velocity                    (time, hybrid, latitude, longitude) float32 753TB ...
    vorticity                            (time, hybrid, latitude, longitude) float32 753TB ...
Attributes:
    last_updated:      2024-08-27 14:20:51.626245
    valid_time_start:  1940-01-01
    valid_time_stop:   2024-03-31

In [11]:
path_weatherbench2 = "gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr"
ds_bench2 = import_data(path_weatherbench2)
ds_bench2


<xarray.Dataset> Size: 669TB
Dimensions:                                           (time: 561264,
                                                       latitude: 721,
                                                       longitude: 1440,
                                                       level: 37)
Coordinates:
  * latitude                                          (latitude) float32 3kB ...
  * level                                             (level) int64 296B 1 .....
  * longitude                                         (longitude) float32 6kB ...
  * time                                              (time) datetime64[ns] 4MB ...
Data variables: (12/48)
    10m_u_component_of_wind                           (time, latitude, longitude) float32 2TB ...
    10m_v_component_of_wind                           (time, latitude, longitude) float32 2TB ...
    2m_dewpoint_temperature                           (time, latitude, longitude) float32 2TB ...
    2m_temperature                                    (time, latitude, longitude) float32 2TB ...
    angle_of_sub_gridscale_orography                  (latitude, longitude) float32 4MB ...
    anisotropy_of_sub_gridscale_orography             (latitude, longitude) float32 4MB ...
    ...                                                ...
    v_component_of_wind                               (time, level, latitude, longitude) float32 86TB ...
    vertical_velocity                                 (time, level, latitude, longitude) float32 86TB ...
    volumetric_soil_water_layer_1                     (time, latitude, longitude) float32 2TB ...
    volumetric_soil_water_layer_2                     (time, latitude, longitude) float32 2TB ...
    volumetric_soil_water_layer_3                     (time, latitude, longitude) float32 2TB ...
    volumetric_soil_water_layer_4                     (time, latitude, longitude) float32 2TB ...